In [1]:
import os
import sys
import ctypes
import resource
import time
import json
import numpy as np

# 0. CUDA HACK
cuda_root = "/oscar/rt/9.6/25/spack/x86_64_v3/cuda-12.9.0-cinrl2oeqemd3szbcakkugp2vtk2fh5t"
os.environ['CUDA_HOME'] = cuda_root
os.environ['CPATH'] = os.path.join(cuda_root, 'include')
os.environ['PATH'] = os.path.join(cuda_root, 'bin') + ":" + os.environ.get('PATH', '')
os.environ['NUMBA_CUDA_DRIVER'] = "/lib64/libcuda.so"
os.environ['LD_LIBRARY_PATH'] = f"{os.path.join(cuda_root, 'nvvm', 'lib64')}:{os.path.join(cuda_root, 'targets', 'x86_64-linux', 'lib')}:{os.path.join(cuda_root, 'lib64')}:/lib64:" + os.environ.get('LD_LIBRARY_PATH', '')
os.environ["IPIE_USE_GPU"] = "1"

try: ctypes.CDLL(os.path.join(cuda_root, "nvvm", "lib64", "libnvvm.so"), mode=ctypes.RTLD_GLOBAL)
except: pass

from pyscf import gto, scf, lib
from ipie.utils.from_pyscf import gen_ipie_input_from_pyscf_chk
from ipie.qmc.afqmc import AFQMC
from ipie.utils.mpi import MPIHandler
import ipie.estimators.local_energy_sd
from ipie.analysis.autocorr import reblock_by_autocorr
from ipie.analysis.extraction import extract_observable

try:
    import cupy as cp
    has_cupy = True
except: has_cupy = False

# CONFIG
N_ATOMS = 80
SYSTEM_NAME = f"H{N_ATOMS}"
WALKERS = 2048
TEST_SEED = 999
NUM_BLOCKS = 50
STEPS_PER_BLOCK = 1

comm = MPIHandler()
rank = comm.rank
if has_cupy and cp.cuda.runtime.getDeviceCount() > 0: cp.cuda.Device(rank % cp.cuda.runtime.getDeviceCount()).use()

chk_file, ham_file, wfn_file = f"scf_h{N_ATOMS}.chk", f"ham_h{N_ATOMS}.h5", f"wfn_h{N_ATOMS}.h5"

if rank == 0:
    mol = gto.M(atom=[("H", 0.74 * j, 0, 0) for j in range(N_ATOMS)], basis="sto-6g", verbose=0)
    mf = scf.UHF(mol)
    mf.chkfile = chk_file
    if not os.path.exists(chk_file) or not os.path.exists(wfn_file):
        mf.kernel()
        gen_ipie_input_from_pyscf_chk(chk_file, hamil_file=ham_file, wfn_file=wfn_file, verbose=0, chol_cut=1e-5)

comm.comm.Barrier()

afqmc = AFQMC.build_from_hdf5(num_elec=(N_ATOMS//2, N_ATOMS//2), ham_file=ham_file, wfn_file=wfn_file, num_blocks=NUM_BLOCKS, num_steps_per_block=STEPS_PER_BLOCK, num_walkers=WALKERS, seed=TEST_SEED, verbose=0)
if has_cupy: afqmc.cuda = True
afqmc.mpi_handler = comm

# Fixed: Always calculate and set nwalkers regardless of comm.size
local_walkers = WALKERS // comm.size
if rank < (WALKERS % comm.size): local_walkers += 1
afqmc.nwalkers = local_walkers

# =============================================================================
# MICRO-BENCHMARKED CLASSICAL INTERCEPT (MODULE LEVEL)
# =============================================================================
baseline_tracker = {"total_time_sec": 0.0, "calls": 0}

# Create a timer wrapper factory
def make_timed_func(orig_func):
    def timed_local_energy(system, hamiltonian, walkers, trial):
        if has_cupy: cp.cuda.Stream.null.synchronize()
        t0 = time.perf_counter()
        
        res = orig_func(system, hamiltonian, walkers, trial)
        
        if has_cupy: cp.cuda.Stream.null.synchronize()
        t1 = time.perf_counter()
        
        baseline_tracker["total_time_sec"] += (t1 - t0)
        baseline_tracker["calls"] += 1
        return res
    return timed_local_energy

# Dynamically patch the global modules (exactly like the ML script)
target_names = [
    "local_energy_single_det_uhf", 
    "local_energy_single_det_batch_gpu", 
    "local_energy_single_det_uhf_batch_gpu",
    "local_energy_single_det_uhf_batch"
]

for name in target_names:
    if hasattr(ipie.estimators.local_energy_sd, name):
        original_func = getattr(ipie.estimators.local_energy_sd, name)
        timed_func = make_timed_func(original_func)
        
        # Patch the local_energy_sd module
        setattr(ipie.estimators.local_energy_sd, name, timed_func)
        
        # Catch any other modules that already imported it
        for mod_name, module in list(sys.modules.items()):
            if module and mod_name.startswith("ipie"):
                try:
                    for attr_name, attr_value in vars(module).items():
                        if attr_value is original_func: 
                            setattr(module, attr_name, timed_func)
                except: pass

# =============================================================================
# RUN
# =============================================================================
if rank == 0: 
    print("\n" + "#"*60 + "\n### STARTING CLASSICAL IPIE GPU PRODUCTION RUN ###\n" + "#"*60)

afqmc.run()

# =============================================================================
# METRICS & EXTRACTION
# =============================================================================
if rank == 0:
    qmc_data = extract_observable(afqmc.estimators.filename, "energy")
    df_ac = reblock_by_autocorr(qmc_data["ETotal"][1:], verbose=0)
    final_energy, final_error = float(df_ac["ETotal_ac"].iloc[0]), float(df_ac["ETotal_error_ac"].iloc[0])

    # ⏱️ Micro-Benchmark Calculations
    avg_loop_time = baseline_tracker["total_time_sec"] / baseline_tracker["calls"] if baseline_tracker["calls"] > 0 else 0
    
    # Analytical estimate of Classical Local Energy FLOPs (O(Nw * N_chol * N_basis^2))
    n_basis = N_ATOMS
    n_chol_approx = n_basis * 5 
    flops_physics_per_walker = 2 * n_chol_approx * (n_basis ** 2)
    flops_total_per_call = flops_physics_per_walker * afqmc.nwalkers
    loop_tflops = (flops_total_per_call / avg_loop_time) / 1e12 if avg_loop_time > 0 else 0

    metrics = {
        "system": SYSTEM_NAME, "backend": "GPU (Classical)",
        "results": {"final_energy_ha": final_energy, "final_error_ha": final_error},
        "micro_benchmark": {
            "total_calls": baseline_tracker["calls"],
            "avg_time_per_call_sec": round(avg_loop_time, 6),
            "est_flops_per_call": flops_total_per_call,
            "throughput_tflops": round(loop_tflops, 6)
        },
        "memory": {
            "peak_gpu_vram_mb": round(cp.get_default_memory_pool().total_bytes() / (1024**2), 2) if has_cupy else 0
        }
    }
    with open(f"micro_metrics_Baseline_{SYSTEM_NAME}.json", "w") as f: json.dump(metrics, f, indent=4)
    print(json.dumps(metrics, indent=4))

# random seed is 999

############################################################
### STARTING CLASSICAL IPIE GPU PRODUCTION RUN ###
############################################################
            Block                   Weight            WeightFactor            HybridEnergy                  ENumer                  EDenom                  ETotal                  E1Body                  E2Body
                0   2.0480000000000000e+03  2.0480000000000000e+03  0.0000000000000000e+00 -8.1836218316002807e+04  2.0480000000000000e+03 -3.9959090974610746e+01 -2.5724417313427648e+02  2.1728508215966573e+02
                1   2.1573156738588791e+03  2.0480000000000000e+03 -2.0804057489548516e+01 -8.6224645389285899e+04  2.1573156738588791e+03 -3.9968487891738320e+01 -2.5724418039844278e+02  2.1727569250670447e+02
                2   2.1572372418155064e+03  2.0480000000000000e+03 -2.0789698824357682e+01 -8.6241648319970205e+04  2.1572372418155064e+03 -3.9977822859849304e+01 -2.572441